# Huấn luyện YOLO11s nhận diện biển báo Việt Nam

Notebook dùng đúng Roboflow workspace **vietnamtrafficsign**, project **vietnam-traffic-sign**, version **1**, định dạng **yolov11**.

Chạy các ô lần lượt từ trên xuống. Nên dùng Google Colab: **Runtime → Change runtime type → GPU**. Cũng có thể chạy bằng Jupyter trên máy cá nhân.

Ta **fine-tune trọng số có sẵn `yolo11s.pt`** trên biển báo của bạn. Đổi thành `yolo11n.pt` nếu cần mô hình nhẹ hơn. Định dạng tải `yolov11` là định dạng dữ liệu; mô hình thực tế được chọn ở bước cấu hình.

Notebook gồm: cài thư viện → tải dữ liệu → kiểm tra nhãn → train → đánh giá → dự đoán → lưu kết quả. Không cần nhập lại tên lớp bằng tay.

## 1. Cài thư viện
`%pip` cài vào đúng môi trường Python của notebook. Nếu notebook yêu cầu khởi động lại kernel sau khi cài, hãy khởi động lại rồi chạy tiếp từ bước 2.

In [ ]:
%pip install -q ultralytics roboflow pyyaml matplotlib pillow

## 2. Kiểm tra thiết bị
Có GPU NVIDIA và PyTorch hỗ trợ CUDA thì dùng GPU số 0; nếu không, dùng CPU. Train trên CPU sẽ chậm. Trên Colab, hãy bật GPU trước khi train.

In [ ]:
from pathlib import Path
from getpass import getpass
import os
import random
import shutil
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display
from ultralytics import YOLO
from roboflow import Roboflow
import ultralytics
import importlib.metadata

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Thiết bị:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("Roboflow:", importlib.metadata.version("roboflow"))

# Mọi dữ liệu và kết quả nằm trong thư mục này.
WORK_DIR = (Path.cwd() / "yolo_bien_bao").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục làm việc:", WORK_DIR)

## 3. Tải dataset Roboflow
Nhập API key tại hộp nhập ẩn. Có thể đặt biến môi trường `ROBOFLOW_API_KEY` trước khi chạy để bỏ qua hộp nhập. Không lưu API key vào notebook hoặc Git.

Giữ nguyên version 1 để các lần thử dùng cùng một bộ dữ liệu. Cần Internet để tải dataset và trọng số pretrained.

In [ ]:
api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass("Nhập Roboflow API key: ")
if not api_key.strip():
    raise ValueError("Bạn chưa nhập API key.")

rf = Roboflow(api_key=api_key)
project = rf.workspace("vietnamtrafficsign").project("vietnam-traffic-sign")
version = project.version(1)
dataset = version.download("yolov11", location=str(WORK_DIR / "dataset"))
DATASET_DIR = Path(dataset.location).resolve()
print("Dataset đã tải về:", DATASET_DIR)

## 4. Đọc danh sách lớp và chuẩn hóa đường dẫn
Roboflow có thể xuất đường dẫn tương đối trong `data.yaml`. Ta tạo **file cấu hình mới** trỏ tới các thư mục đã tải để tránh lỗi tìm ảnh. File gốc được giữ nguyên. Không tự chia lại các tập dữ liệu.

Nếu Roboflow đổi cấu trúc thư mục, ô này sẽ dừng và báo rõ thay vì train nhầm dữ liệu.

In [ ]:
ORIGINAL_YAML = DATASET_DIR / "data.yaml"
with ORIGINAL_YAML.open(encoding="utf-8") as f:
    original_config = yaml.safe_load(f)

names = original_config["names"]
if isinstance(names, list):
    CLASS_NAMES = {i: name for i, name in enumerate(names)}
else:
    CLASS_NAMES = {int(i): name for i, name in names.items()}
assert sorted(CLASS_NAMES) == list(range(len(CLASS_NAMES))), "ID lớp phải liên tục từ 0."

TRAIN_IMAGES = DATASET_DIR / "train" / "images"
VAL_IMAGES = DATASET_DIR / "valid" / "images"
TEST_IMAGES = DATASET_DIR / "test" / "images"
for folder in (TRAIN_IMAGES, VAL_IMAGES):
    if not folder.is_dir():
        raise FileNotFoundError(f"Không tìm thấy {folder}. Hãy kiểm tra thư mục dataset.")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def get_images(folder):
    return sorted(p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)

assert get_images(TRAIN_IMAGES), "Tập train không có ảnh."
assert get_images(VAL_IMAGES), "Tập validation không có ảnh."
HAS_TEST = TEST_IMAGES.is_dir() and bool(get_images(TEST_IMAGES))

config = {
    "path": str(DATASET_DIR),
    "train": str(TRAIN_IMAGES),
    "val": str(VAL_IMAGES),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
if HAS_TEST:
    config["test"] = str(TEST_IMAGES)

DATA_YAML = DATASET_DIR / "data_local.yaml"
with DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)

print("Số lớp:", len(CLASS_NAMES))
for class_id, class_name in CLASS_NAMES.items():
    print(f"  {class_id}: {class_name}")
print("Số ảnh train:", len(get_images(TRAIN_IMAGES)))
print("Số ảnh validation:", len(get_images(VAL_IMAGES)))
print("Số ảnh test:", len(get_images(TEST_IMAGES)) if HAS_TEST else "Không có")
print("Cấu hình dùng để train:", DATA_YAML)

## 5. Kiểm tra nhãn train và xem ảnh có bounding box
Mỗi dòng nhãn detection YOLO có dạng:

`class_id x_center y_center width height`

Bốn tọa độ được chuẩn hóa theo kích thước ảnh. Ảnh không có biển có thể có file nhãn rỗng hoặc không có file nhãn. Nếu số ảnh thiếu nhãn lớn bất thường, kiểm tra lại trước khi train.

Ta kiểm tra toàn bộ nhãn train và vẽ thử một số ảnh có biển để bạn kiểm tra bằng mắt.

In [ ]:
train_images = get_images(TRAIN_IMAGES)
class_counts = {class_id: 0 for class_id in CLASS_NAMES}
missing_labels = 0
empty_labels = 0
labeled_images = []

def label_path(image_path):
    # Giữ cấu trúc thư mục con nếu dataset có thư mục con.
    relative = image_path.relative_to(TRAIN_IMAGES)
    return DATASET_DIR / "train" / "labels" / relative.with_suffix(".txt")

for image_path in train_images:
    annotation = label_path(image_path)
    if not annotation.exists():
        missing_labels += 1
        continue
    lines = annotation.read_text(encoding="utf-8").splitlines()
    lines = [line for line in lines if line.strip()]
    if not lines:
        empty_labels += 1
        continue
    labeled_images.append(image_path)
    for line_number, line in enumerate(lines, 1):
        values = line.split()
        assert len(values) == 5, f"Sai định dạng: {annotation}, dòng {line_number}"
        class_value, x, y, w, h = map(float, values)
        class_id = int(class_value)
        assert class_value == class_id and class_id in CLASS_NAMES, f"Sai ID lớp: {annotation}"
        assert all(0 <= v <= 1 for v in (x, y, w, h)) and w > 0 and h > 0, f"Sai tọa độ: {annotation}"
        class_counts[class_id] += 1

assert labeled_images, "Không có ảnh train nào chứa nhãn biển báo hợp lệ."
print("Ảnh thiếu file nhãn:", missing_labels)
print("Ảnh có file nhãn rỗng:", empty_labels)
print("Số bounding box theo lớp:")
for class_id, count in class_counts.items():
    print(f"  {CLASS_NAMES[class_id]}: {count}")

sample_images = random.Random(42).sample(labeled_images, min(4, len(labeled_images)))
for image_path in sample_images:
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for line in label_path(image_path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        class_id, x, y, w, h = map(float, line.split())
        box = ((x - w/2)*width, (y - h/2)*height, (x + w/2)*width, (y + h/2)*height)
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0], max(0, box[1] - 14)), str(int(class_id)), fill="red")
    print(image_path.name, "— số trên khung là ID lớp trong danh sách phía trên")
    image.thumbnail((900, 600))
    display(image)

## 6. Cấu hình huấn luyện
- `MODEL_NAME`: YOLO11s pretrained; đổi sang `yolo11n.pt` để nhẹ hơn.
- `EPOCHS=100`: tối đa 100 lượt học qua tập train.
- `IMAGE_SIZE=640`: kích thước đầu vào; thử 960 nếu biển nhỏ hay bị bỏ sót, nhưng sẽ tốn bộ nhớ hơn.
- `BATCH_SIZE=8`: số ảnh trong một bước cập nhật. Nếu lỗi **CUDA out of memory**, giảm xuống 4 hoặc 2 rồi chạy lại ô train.
- `PATIENCE=20`: dừng sớm khi chỉ số validation không cải thiện trong 20 epoch.

Tắt lật ngang/dọc vì lật biển rẽ trái có thể làm ảnh giống biển rẽ phải trong khi nhãn vẫn giữ nguyên. `workers=0` giúp notebook dễ chạy cả trên Windows.

In [ ]:
MODEL_NAME = "yolo11s.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 8 if DEVICE == 0 else 2
PATIENCE = 20
RUNS_DIR = WORK_DIR / "runs"

model = YOLO(MODEL_NAME)
print("Mô hình:", MODEL_NAME)
print("Epochs:", EPOCHS, "| Image size:", IMAGE_SIZE, "| Batch:", BATCH_SIZE)

## 7. Bắt đầu train
Ô này có thể chạy lâu. Lần đầu, Ultralytics tải trọng số pretrained. Mỗi epoch hiển thị loss và chỉ số validation.

Kết quả được lưu trong `yolo_bien_bao/runs`. Mỗi lần chạy lại tạo thư mục mới nếu tên cũ đã tồn tại. `best.pt` được chọn theo fitness trên validation của Ultralytics; `last.pt` dùng để tiếp tục lượt train bị gián đoạn.

In [ ]:
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=0,
    patience=PATIENCE,
    optimizer="auto",
    seed=42,
    deterministic=True,
    fliplr=0.0,
    flipud=0.0,
    close_mosaic=10,
    project=str(RUNS_DIR),
    name="yolo11_bien_bao",
    exist_ok=False,
    save=True,
    plots=True,
)

# Lấy đường dẫn thực tế, kể cả khi Ultralytics tự thêm số vào tên thư mục.
RUN_DIR = Path(model.trainer.save_dir).resolve()
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
assert BEST_PT.exists(), f"Chưa tìm thấy trọng số: {BEST_PT}"
print("Thư mục kết quả:", RUN_DIR)
print("Trọng số tốt nhất:", BEST_PT)

## 8. Xem biểu đồ kết quả
`results.png` thể hiện loss và chỉ số qua các epoch. Ma trận nhầm lẫn giúp tìm những lớp hay bị nhận nhầm. Đây là kết quả validation của lượt train.

In [ ]:
for filename in ["results.png", "confusion_matrix_normalized.png"]:
    path = RUN_DIR / filename
    if path.exists():
        print(filename)
        display(Image.open(path))

## 9. Đánh giá best.pt trên validation
- **Precision**: trong các dự đoán là biển, bao nhiêu dự đoán đúng.
- **Recall**: trong các biển thật, mô hình tìm được bao nhiêu.
- **mAP50** và **mAP50–95**: chỉ số tổng hợp detection; càng cao càng tốt.

Đây là chỉ số validation, không phải kết quả test độc lập. Không đặt `conf=0.25` ở bước tính mAP để giữ quy trình đánh giá mặc định của Ultralytics.

In [ ]:
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=0,
    project=str(RUNS_DIR),
    name="danh_gia_validation",
    plots=True,
)
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")
print(f"mAP50:     {val_metrics.box.map50:.4f}")
print(f"mAP50-95:  {val_metrics.box.map:.4f}")

## 10. Đánh giá test độc lập nếu dataset có tập test
Chỉ dùng test để báo cáo kết quả sau khi đã chốt cấu hình bằng validation. Không liên tục chọn cấu hình dựa trên test. Nếu không có test, notebook bỏ qua và báo rõ.

In [ ]:
if HAS_TEST:
    test_metrics = best_model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=0,
        project=str(RUNS_DIR),
        name="danh_gia_test",
        plots=True,
    )
    print(f"Test mAP50:    {test_metrics.box.map50:.4f}")
    print(f"Test mAP50-95: {test_metrics.box.map:.4f}")
else:
    print("Dataset không có tập test. Chỉ mới có kết quả validation.")

## 11. Dự đoán thử trên ảnh chưa dùng để cập nhật trọng số
Ưu tiên ảnh test; nếu không có thì dùng validation. `conf=0.25` là ngưỡng hiển thị dự đoán: tăng để bớt dự đoán yếu, giảm để xem thêm các biển mô hình chưa chắc chắn.

In [ ]:
preview_folder = TEST_IMAGES if HAS_TEST else VAL_IMAGES
preview_images = get_images(preview_folder)[:4]
predictions = best_model.predict(
    source=[str(p) for p in preview_images],
    imgsz=IMAGE_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(RUNS_DIR),
    name="du_doan_mau",
)
for result in predictions:
    # plot() trả ảnh BGR; đổi sang RGB để matplotlib hiển thị đúng màu.
    plt.figure(figsize=(12, 7))
    plt.imshow(result.plot()[:, :, ::-1])
    plt.axis("off")
    plt.title(Path(result.path).name)
    plt.show()

## 12. Lưu mô hình và đóng gói kết quả
`bien_bao_best.pt` là file dùng để tích hợp vào chương trình detect. File ZIP chứa trọng số, cấu hình dữ liệu, phiên bản thư viện và kết quả train; không chứa toàn bộ dataset.

Trên Colab, dữ liệu sẽ mất khi phiên bị xóa. Hãy tải ZIP về sau khi train hoặc lưu kết quả vào Google Drive.

In [ ]:
EXPORT_DIR = WORK_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = EXPORT_DIR / "bien_bao_best.pt"
shutil.copy2(BEST_PT, MODEL_PATH)
shutil.copy2(DATA_YAML, EXPORT_DIR / DATA_YAML.name)

versions = {
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "roboflow": importlib.metadata.version("roboflow"),
}
(EXPORT_DIR / "versions.txt").write_text(
    "\n".join(f"{name}=={value}" for name, value in versions.items()), encoding="utf-8"
)
shutil.copy2(EXPORT_DIR / "versions.txt", RUN_DIR / "versions.txt")
shutil.copy2(DATA_YAML, RUN_DIR / DATA_YAML.name)
ZIP_PATH = shutil.make_archive(str(WORK_DIR / f"{RUN_DIR.name}_ket_qua"), "zip", root_dir=RUN_DIR)
print("Mô hình để sử dụng:", MODEL_PATH)
print("Kết quả đóng gói:", ZIP_PATH)

# Trên Google Colab, bỏ dấu # ở hai dòng dưới để tải ZIP về máy.
# from google.colab import files
# files.download(ZIP_PATH)

## 13. Dùng ảnh/video riêng hoặc tiếp tục lượt train bị gián đoạn (tùy chọn)
Các lệnh dưới đang được chú thích để không chạy ngoài ý muốn. Thay đường dẫn và bỏ dấu `#` khi cần.

Để resume sau khi khởi động lại kernel, import `YOLO`, chỉ rõ đường dẫn `last.pt` và bảo đảm dataset vẫn ở vị trí cũ. Resume dành cho lượt train **chưa hoàn thành**, không phải tăng epoch của một lượt đã hoàn tất.

In [ ]:
# Dự đoán ảnh riêng:
# detector = YOLO(str(MODEL_PATH))
# detector.predict(source="duong_dan/anh.jpg", conf=0.25, save=True)

# Dự đoán video riêng:
# detector = YOLO(str(MODEL_PATH))
# for result in detector.predict(source="duong_dan/video.mp4", conf=0.25, save=True, stream=True):
#     pass  # stream=True giúp không giữ kết quả của cả video trong RAM.

# Tiếp tục một lượt train bị gián đoạn:
# resumed_model = YOLO("duong_dan/runs/yolo11_bien_bao/weights/last.pt")
# resumed_model.train(resume=True, workers=0)

## Ghi chú khi đọc kết quả
- Kiểm tra nhãn, ảnh thực tế và các lớp ít mẫu trước khi đổi mô hình lớn hơn.
- Các frame gần giống nhau có thể gây kết quả quá lạc quan nếu nằm ở cả train và test. Notebook dùng cách chia sẵn của Roboflow; cần tự kiểm tra trùng lặp/nguồn ảnh trước khi báo cáo.
- Dataset đường phố thật có thể không phù hợp trực tiếp với mô phỏng VIA. Nếu chạy trong mô phỏng, cần đánh giá và bổ sung ảnh từ chính mô phỏng.

Tài liệu: [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/), [Train](https://docs.ultralytics.com/modes/train/), [Roboflow SDK](https://docs.roboflow.com/reference/platform/python-sdk).